# BirdCLEF+ 2026 - Baseline Model

## Score: .480

In [ ]:
import numpy as np
import pandas as pd
import torch
from scipy.ndimage import zoom
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path("/kaggle/input/competitions/birdclef-2026")

In [ ]:
# Load metadata
train_df = pd.read_csv(DATA_DIR / "train.csv")
taxonomy_df = pd.read_csv(DATA_DIR / "taxonomy.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Species list (submission columns)
SPECIES = [c for c in sample_sub.columns if c != "row_id"]
NUM_CLASSES = len(SPECIES)
species_to_idx = {s: i for i, s in enumerate(SPECIES)}

print(f"Species: {NUM_CLASSES}")
print(f"Train samples: {len(train_df)}")

In [ ]:
# Config
SR = 32000  # 32 kHz
DURATION = 5.0
N_MELS = 128
N_FFT = 2048
HOP_LEN = 512
IMG_SIZE = (128, 128)  # mel bins x time frames
BATCH_SIZE = 32
EPOCHS = 3  # Keep low for CPU / 90 min limit
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
def load_audio_segment(filepath, sr=SR, duration=DURATION):
    """Load audio and extract center segment or pad/trim to duration."""
    y, _ = librosa.load(filepath, sr=sr, mono=True)
    target_len = int(sr * duration)
    if len(y) >= target_len:
        start = (len(y) - target_len) // 2
        y = y[start : start + target_len]
    else:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    return y


def audio_to_mel(y, n_mels=N_MELS, n_fft=N_FFT, hop_len=HOP_LEN):
    """Compute log-mel spectrogram and resize to IMG_SIZE."""
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=n_mels, n_fft=n_fft, hop_length=hop_len)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # Resize to fixed size
    h, w = mel_db.shape
    zoom_factors = (IMG_SIZE[0] / h, IMG_SIZE[1] / w)
    mel_resized = zoom(mel_db, zoom_factors, order=1)
    return mel_resized.astype(np.float32)

In [ ]:
class BirdAudioDataset(Dataset):
    def __init__(self, df, audio_dir, species_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.species_to_idx = species_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = self.audio_dir / row["filename"]
        label = row["primary_label"]
        label_idx = self.species_to_idx.get(label, -1)
        if label_idx < 0:
            label_idx = 0  # fallback
        y = load_audio_segment(str(filepath))
        mel = audio_to_mel(y)
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)  # normalize 0-1
        x = torch.from_numpy(mel).unsqueeze(0).float()  # (1, H, W)
        return x, torch.tensor(label_idx, dtype=torch.long)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

In [ ]:
# Filter train to species in submission, subsample for speed
train_filtered = train_df[train_df["primary_label"].isin(SPECIES)].copy()
if len(train_filtered) > 15000:
    train_filtered = train_filtered.sample(15000, random_state=42)

train_ds = BirdAudioDataset(train_filtered, DATA_DIR / "train_audio", species_to_idx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)

model = SimpleCNN(NUM_CLASSES).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# Inference on test soundscapes
test_dir = DATA_DIR / "test_soundscapes"
soundscape_files = sorted(test_dir.glob("*.ogg"))

model.eval()
rows = []
with torch.no_grad():
    for fpath in tqdm(soundscape_files, desc="Predicting"):
        fname = fpath.stem
        y, _ = librosa.load(str(fpath), sr=SR, mono=True)
        seg_len = int(SR * DURATION)
        for start in range(0, len(y) - seg_len + 1, seg_len):
            seg = y[start : start + seg_len]
            mel = audio_to_mel(seg)
            mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)
            x = torch.from_numpy(mel).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            end_time = start // SR + 5
            row_id = f"{fname}_{end_time}"
            rows.append({"row_id": row_id, **dict(zip(SPECIES, probs))})
sub = pd.DataFrame(rows)[["row_id"] + SPECIES]
sub.to_csv("submission.csv", index=False)
print(sub.head())